# Import data

In [1]:
# ── Cell 1 — Config ───────────────────────────────────────────────────────────

panels = {
    "CGN":   ["C3","CD46","CFH","CFHR5","CFI","COL4A3","COL4A4","COL4A5","COL4A6","FN1"],
    "CAKUT": ["ACE","AGT","AGTR1","BMP4","CHD1L","CHRM3","DSTYK","EYA1","FGF20","FRAS1",
              "FREM1","FREM2","GATA3","GRIP1","HNF1B","HPSE2","ITGA8","KAL1","LRIG2","MUC1",
              "PAX2","REN","RET","ROBO2","SALL1","SIX1","SIX2","SIX5","SOX17","SRGAP1",
              "TBX18","TNXB","TRAP1","UMOD","UPK3A","WNT4"],
    "SRNS":  ["ACTN4","ADCK4","ANLN","ARHGAP24","ARHGDIA","CD2AP","COQ2","COQ6","CRB2",
          "CUBN","DGKE","EMP2","FAT1","INF2","ITGA3","ITGB4","KANK1","KANK2","KANK4",
          "LAMB2","LMX1B","MTTL1","MYH9","MYO1E","NPHS1","NPHS2","NUP107","NUP205",
          "NUP93","PDSS2","PLCE1","PTPRO","SCARB2","SMARCAL1","TRPC6","WDR73","WT1","XPO5"],
    "USD":   ["ADCY10","AGXT","APRT","ATP6V0A4","ATP6V1B1","CA2","CASR","CLCN5","CLCNKB",
              "CLDN16","CLDN19","CYP24A1","FAM20A","GRHPR","HNF4A","HOGA1","HPRT1","KCNJ1",
              "OCRL","SLC12A1","SLC22A12","SLC2A9","SLC34A1","SLC34A3","SLC3A1","SLC4A1",
              "SLC7A9","SLC9A3R1","VDR","XDH"],
    "NPHP":  ["ANKS6","CEP164","CEP290","GLIS2","INVS","IQCB1","NEK8","NPHP1","NPHP3",
              "NPHP4","RPGRIP1L","SDCCAG8","TMEM67","TTC21B","WDR19","ZNF423"],
}

gene_panel   = {gene: panel for panel, genes in panels.items() for gene in genes}
all_genes    = list(gene_panel.keys())

rsa_base     = "https://raw.githubusercontent.com/NephVar/NephVar/main/biophysical"
clinvar_base = "https://raw.githubusercontent.com/Joshua-Pillai/NephVar/main"

buried_threshold = 0.25

PATHOGENIC = {"Pathogenic","Likely pathogenic","Pathogenic/Likely pathogenic",
              "Pathogenic, low penetrance","Likely pathogenic, low penetrance",
              "Pathogenic/Likely pathogenic, low penetrance",
              "Likely pathogenic/Likely pathogenic, low penetrance",
              "Likely pathogenic/Pathogenic, low penetrance"}
VUS_SET    = {"Uncertain significance","Uncertain significance/Uncertain risk allele",
              "Uncertain risk allele","Likely pathogenic/Likely risk allele",
              "Likely risk allele","Uncertain significance/VUS-mid"}
BENIGN     = {"Benign","Likely benign","Benign/Likely benign"}

def bucket(label):
    if pd.isna(label):
        return "Other"
    label = str(label).strip()
    if label in PATHOGENIC:
        return "P" if label == "Pathogenic" else "LP"
    if label in VUS_SET:
        return "VUS"
    if label in BENIGN:
        return "B" if label == "Benign" else "LB"
    base = label.split(";")[0].strip()
    if base == "Pathogenic":         return "P"
    if base == "Likely pathogenic":  return "LP"
    if base == "Benign":             return "B"
    if base == "Likely benign":      return "LB"
    if "VUS" in label or "Uncertain significance" in label:
        return "VUS"
    return "Other"

In [2]:
# ── Cell 1b — MOI annotation for 129 NephVar genes ───────────────────────────
import pandas as pd
from collections import Counter

MOI = {
    # CGN
    "C3":"AD_AR","CD46":"AD_AR","CFH":"AR","CFHR5":"AD","CFI":"AD",
    "COL4A3":"AD_AR","COL4A4":"AD_AR","COL4A5":"XL","COL4A6":"XL","FN1":"AD",
    # CAKUT
    "ACE":"AR","AGT":"AR","AGTR1":"AR","BMP4":"AD","CHD1L":"AD","CHRM3":"AR",
    "DSTYK":"AD","EYA1":"AD","FGF20":"AR","FRAS1":"AR","FREM1":"AR","FREM2":"AR",
    "GATA3":"AD","GRIP1":"AR","HNF1B":"AD","HPSE2":"AR","ITGA8":"AR","KAL1":"XL",
    "LRIG2":"AR","MUC1":"AD","PAX2":"AD","REN":"AR","RET":"AD","ROBO2":"AD",
    "SALL1":"AD","SIX1":"AD","SIX2":"AD","SIX5":"AD","SOX17":"AD","SRGAP1":"AD",
    "TBX18":"AD","TNXB":"AD","TRAP1":"AR","UMOD":"AD","UPK3A":"AD","WNT4":"AD",
    # SRNS
    "ACTN4":"AD","ADCK4":"AR","ANLN":"AD","ARHGAP24":"AD","ARHGDIA":"AR",
    "CD2AP":"AR","COQ2":"AR","COQ6":"AR","CRB2":"AR","CUBN":"AR","DGKE":"AR",
    "EMP2":"AR","FAT1":"AR","INF2":"AD","ITGA3":"AR","ITGB4":"AR","KANK1":"AR",
    "KANK2":"AR","KANK4":"AR","LAMB2":"AR","LMX1B":"AD","MTTL1":None,"MYH9":"AD",
    "MYO1E":"AR","NPHS1":"AR","NPHS2":"AR","NUP107":"AR","NUP205":"AR","NUP93":"AR",
    "PDSS2":"AR","PLCE1":"AR","PTPRO":"AR","SCARB2":"AR","SMARCAL1":"AR","TRPC6":"AD",
    "WDR73":"AR","WT1":"AD","XPO5":"AR",
    # USD
    "ADCY10":"AD","AGXT":"AR","APRT":"AR","ATP6V0A4":"AR","ATP6V1B1":"AR",
    "CA2":"AR","CASR":"AD","CLCN5":"XL","CLCNKB":"AR","CLDN16":"AR","CLDN19":"AR",
    "CYP24A1":"AR","FAM20A":"AR","GRHPR":"AR","HNF4A":"AD","HOGA1":"AR","HPRT1":"XL",
    "KCNJ1":"AR","OCRL":"XL","SLC12A1":"AR","SLC22A12":"AD_AR","SLC2A9":"AD_AR",
    "SLC34A1":"AD_AR","SLC34A3":"AR","SLC3A1":"AR","SLC4A1":"AD_AR","SLC7A9":"AD_AR",
    "SLC9A3R1":"AD","VDR":"AD","XDH":"AR",
    # NPHP — all AR
    "ANKS6":"AR","CEP164":"AR","CEP290":"AR","GLIS2":"AR","INVS":"AR","IQCB1":"AR",
    "NEK8":"AR","NPHP1":"AR","NPHP3":"AR","NPHP4":"AR","RPGRIP1L":"AR","SDCCAG8":"AR",
    "TMEM67":"AR","TTC21B":"AR","WDR19":"AR","ZNF423":"AR",
}

n_annotated = sum(1 for v in MOI.values() if v is not None)
missing = [g for g in all_genes if g not in MOI]
print(f"Genes in panels : {len(all_genes)}")
print(f"Genes in MOI    : {len(MOI)}  ({n_annotated} with MOI, 1 excluded: MTTL1)")
print(f"Missing from MOI: {missing}")
print()
for moi, n in sorted(Counter(v for v in MOI.values() if v).items()):
    print(f"  {moi:6s}: {n} genes")

Genes in panels : 130
Genes in MOI    : 130  (129 with MOI, 1 excluded: MTTL1)
Missing from MOI: []

  AD    : 37 genes
  AD_AR : 9 genes
  AR    : 77 genes
  XL    : 6 genes


In [3]:
# ── Cell 2 — Load RSA ─────────────────────────────────────────────────────────
import re, json, urllib.request, pandas as pd

rows = []
errors = []

for gene in all_genes:
    url = f"{rsa_base}/{gene}.html"
    try:
        with urllib.request.urlopen(url, timeout=15) as r:
            html = r.read().decode("utf-8", errors="replace")
        m = re.search(r'const residues = (\[.*?\]);', html, re.DOTALL)
        if not m:
            raise ValueError("residues block not found")
        residues = json.loads(m.group(1))
        for res in residues:
            rows.append({
                "gene"  : gene,
                "resnum": res["resnum"],
                "rsa"   : res["rsa"],
                "ss"    : res["ss"],
                "plddt" : res["plddt"],
            })
    except Exception as e:
        errors.append(gene)
        print(f"  ERROR {gene}: {e}")

df_rsa = pd.DataFrame(rows)
print(f"Genes loaded  : {df_rsa['gene'].nunique()} / {len(all_genes)}")
print(f"Total residues: {len(df_rsa):,}")
if errors:
    print(f"Errors: {errors}")

  ERROR MTTL1: HTTP Error 404: Not Found
Genes loaded  : 129 / 130
Total residues: 127,472
Errors: ['MTTL1']


In [4]:
# ── Cell 3 — Load ClinVar missense ────────────────────────────────────────────
import io, re, urllib.request
import pandas as pd

rsa_residues = set(zip(df_rsa['gene'], df_rsa['resnum']))

def parse_resnum(protein_change, gene):
    if pd.isna(protein_change):
        return None
    first_parsed = None
    for part in str(protein_change).split(','):
        part = part.strip()
        m = re.search(r'[A-Za-z*](\d+)', part)
        if m:
            resnum = int(m.group(1))
            if first_parsed is None:
                first_parsed = resnum
            if (gene, resnum) in rsa_residues:
                return resnum
    return first_parsed

rows = []
errors = []

for panel, genes in panels.items():
    for gene in genes:
        url = f"{clinvar_base}/{panel}/{gene}.txt"
        try:
            with urllib.request.urlopen(url, timeout=15) as r:
                content = r.read().decode("utf-8", errors="replace")
            df = pd.read_csv(io.StringIO(content), sep="\t", low_memory=False)
            df["gene"]  = gene
            df["panel"] = panel
            rows.append(df)
        except Exception as e:
            errors.append(gene)
            print(f"  ERROR {gene}: {e}")

df_raw = pd.concat(rows, ignore_index=True)
n_raw  = len(df_raw)

n_dup_rows = df_raw['VariationID'].duplicated(keep='first').sum()
df_raw = df_raw.drop_duplicates(subset='VariationID', keep='first').copy()
print(f"Raw rows: {n_raw:,}  |  Removed {n_dup_rows:,} duplicate VariationIDs  |  Unique: {len(df_raw):,}")
assert len(df_raw) == 117373, f"Expected 117,373 unique variants, got {len(df_raw):,}"

df_miss = df_raw[
    df_raw["Molecular consequence"].str.contains("missense", case=False, na=False)
].copy()

df_miss["resnum"] = df_miss.apply(
    lambda row: parse_resnum(row["Protein change"], row["gene"]), axis=1
)

df_miss["acmg"] = df_miss["Germline classification"].apply(bucket)

n_total   = len(df_miss)
n_parsed  = df_miss["resnum"].notna().sum()
print(f"Total missense   : {n_total:,}")
print(f"Residue parsed   : {n_parsed:,} ({100*n_parsed/n_total:.1f}%)")
print(f"Unparseable      : {n_total - n_parsed}")
print(f"\nACMG distribution:")
print(df_miss["acmg"].value_counts().to_string())
if errors:
    print(f"\nErrors: {errors}")

Raw rows: 119,315  |  Removed 1,942 duplicate VariationIDs  |  Unique: 117,373
Total missense   : 50,462
Residue parsed   : 50,425 (99.9%)
Unparseable      : 37

ACMG distribution:
acmg
VUS    44113
LP      2616
LB      2241
P        899
B        593


In [5]:
# ── Cell 4 — Join RSA to missense variants ────────────────────────────────────
df_miss_clean = df_miss[
    df_miss["resnum"].notna() &
    (df_miss["acmg"] != "Other")
].copy()

df_merged = df_miss_clean.merge(
    df_rsa[["gene", "resnum", "rsa", "ss", "plddt"]],
    on=["gene", "resnum"],
    how="left"
)

n_total = len(df_merged)
n_rsa   = df_merged["rsa"].notna().sum()
n_no_rsa = n_total - n_rsa

print(f"Variants into join  : {n_total:,}")
print(f"RSA matched         : {n_rsa:,} ({100*n_rsa/n_total:.1f}%)")
print(f"No RSA match        : {n_no_rsa:,}")
print(f"\nACMG counts after join:")
print(df_merged[df_merged["rsa"].notna()]["acmg"].value_counts().to_string())

df_rsa_clean = df_merged[df_merged["rsa"].notna()].copy()

Variants into join  : 50,425
RSA matched         : 50,346 (99.8%)
No RSA match        : 79

ACMG counts after join:
acmg
VUS    44021
LP      2609
LB      2230
P        895
B        591


# Cα coordinates

In [6]:
# ── Cell 5 — Load Cα coordinates from GitHub PDB structures ──────────────────
import urllib.request
import numpy as np

STRUCT_BASE = "https://raw.githubusercontent.com/Joshua-Pillai/NephVar/main/Structures"

def load_ca_coords(gene):
    for fname in [f"{gene}_F1.pdb", f"{gene}.pdb"]:
        url = f"{STRUCT_BASE}/{fname}"
        try:
            with urllib.request.urlopen(url, timeout=15) as r:
                pdb = r.read().decode("utf-8", errors="replace")
            coords = {}
            for line in pdb.splitlines():
                if line[:4] in ("ATOM","HETA") and line[12:16].strip() == "CA":
                    resnum = int(line[22:26].strip())
                    x = float(line[30:38].strip())
                    y = float(line[38:46].strip())
                    z = float(line[46:54].strip())
                    coords[resnum] = (x, y, z)
            if coords:
                return coords
        except Exception:
            continue
    return {}

ca_coords = {}
missing_struct = []

for gene in all_genes:
    if gene == "MTTL1":
        continue
    coords = load_ca_coords(gene)
    if coords:
        ca_coords[gene] = coords
    else:
        missing_struct.append(gene)

print(f"Structures loaded : {len(ca_coords)} / {len(all_genes)-1}")
print(f"Missing structures: {missing_struct}")

Structures loaded : 129 / 129
Missing structures: []


# EDC Calculations

In [7]:
# ── Cell 6 — Compute EDC (PLP + BLB) + Export GraphPad CSVs ─────────────────
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, mannwhitneyu

def compute_edc_correct(disease_residues, all_residues_coords):
    """
    EDC per Gerasimavicius et al. 2022 Eq 1-2.
    EDC = mean(log Dmin, non-disease) / mean(log Dmin, disease)
    """
    disease_set    = set(disease_residues)
    disease_coords = np.array([all_residues_coords[r] for r in disease_residues
                                if r in all_residues_coords])
    if len(disease_coords) < 2:
        return None
    all_res    = list(all_residues_coords.keys())
    all_coords = np.array([all_residues_coords[r] for r in all_res])

    disease_log_dmin    = []
    nondisease_log_dmin = []
    for i, r in enumerate(all_res):
        diffs = disease_coords - all_coords[i]
        dists = np.sqrt((diffs**2).sum(axis=1))
        if r in disease_set:
            dists = dists[dists > 0]
        if len(dists) == 0:
            continue
        dmin = dists.min()
        if dmin <= 0:
            continue
        if r in disease_set:
            disease_log_dmin.append(np.log(dmin))
        else:
            nondisease_log_dmin.append(np.log(dmin))

    if len(disease_log_dmin) < 2 or len(nondisease_log_dmin) < 2:
        return None
    mean_d  = np.mean(disease_log_dmin)
    mean_nd = np.mean(nondisease_log_dmin)
    if mean_d == 0:
        return None
    return mean_nd / mean_d

# ── Pre-group df_rsa_clean by gene ───────────────────────────────────────────
rsa_by_gene = {gene: grp for gene, grp in df_rsa_clean.groupby("gene")}

# ── Compute P/LP EDC ─────────────────────────────────────────────────────────
print("Computing P/LP EDC...")
edc_plp_rows = []
for gene in all_genes:
    if gene == "MTTL1" or gene not in ca_coords:
        continue
    moi   = MOI.get(gene)
    panel = gene_panel[gene]
    gene_df = rsa_by_gene.get(gene, pd.DataFrame())
    plp_res = (gene_df[gene_df["acmg"].isin(["P","LP"])]
               ["resnum"].dropna().astype(int).unique().tolist())
    n_plp = len(plp_res)
    if n_plp < 3:
        edc_plp_rows.append({"gene":gene,"panel":panel,"moi":moi,
                              "n_plp_residues":n_plp,"edc":None,"note":"too few P/LP"})
        continue
    edc_plp_rows.append({"gene":gene,"panel":panel,"moi":moi,
                          "n_plp_residues":n_plp,
                          "edc":compute_edc_correct(plp_res, ca_coords[gene]),
                          "note":""})

df_edc = pd.DataFrame(edc_plp_rows)
print(f"  P/LP EDC: {df_edc['edc'].notna().sum()} genes computed")

# ── Compute B/LB EDC ─────────────────────────────────────────────────────────
print("Computing B/LB EDC...")
edc_blb_rows = []
for gene in all_genes:
    if gene == "MTTL1" or gene not in ca_coords:
        continue
    moi   = MOI.get(gene)
    panel = gene_panel[gene]
    gene_df = rsa_by_gene.get(gene, pd.DataFrame())
    blb_res = (gene_df[gene_df["acmg"].isin(["B","LB"])]
               ["resnum"].dropna().astype(int).unique().tolist())
    n_blb = len(blb_res)
    if n_blb < 3:
        edc_blb_rows.append({"gene":gene,"panel":panel,"moi":moi,
                              "n_blb_residues":n_blb,"edc_blb":None,"note":"too few B/LB"})
        continue
    edc_blb_rows.append({"gene":gene,"panel":panel,"moi":moi,
                          "n_blb_residues":n_blb,
                          "edc_blb":compute_edc_correct(blb_res, ca_coords[gene]),
                          "note":""})

df_edc_blb = pd.DataFrame(edc_blb_rows)
print(f"  B/LB EDC: {df_edc_blb['edc_blb'].notna().sum()} genes computed")

# ── Merge ─────────────────────────────────────────────────────────────────────
df_edc_both = df_edc[["gene","panel","moi","n_plp_residues","edc"]].merge(
    df_edc_blb[["gene","n_blb_residues","edc_blb"]],
    on="gene", how="inner"
).dropna(subset=["edc","edc_blb"])
df_edc_both["delta"] = df_edc_both["edc"] - df_edc_both["edc_blb"]
print(f"  Genes with both PLP+BLB EDC: {len(df_edc_both)}")

# ── Stats ─────────────────────────────────────────────────────────────────────
ad = df_edc[df_edc["moi"]=="AD"]["edc"].dropna()
ar = df_edc[df_edc["moi"]=="AR"]["edc"].dropna()
xl = df_edc[df_edc["moi"]=="XL"]["edc"].dropna()
ad_ar = df_edc[df_edc["moi"]=="AD_AR"]["edc"].dropna()

_, p_mw = mannwhitneyu(ad, ar, alternative="greater")
print(f"\nAD vs AR Mann-Whitney p={p_mw:.4f}")
print(f"\nMOI summary (P/LP EDC):")
for moi, grp in [("AD",ad),("AR",ar),("XL",xl),("AD_AR",ad_ar)]:
    print(f"  {moi:<6}: n={len(grp):>3}  median={grp.median():.4f}  "
          f"IQR=[{grp.quantile(0.25):.3f},{grp.quantile(0.75):.3f}]")

print(f"\nPLP vs BLB Wilcoxon by MOI:")
for moi in ["AD","AR","XL","AD_AR"]:
    sub = df_edc_both[df_edc_both["moi"]==moi]
    if len(sub) < 5:
        print(f"  {moi:<6}: n={len(sub)} — n/a")
        continue
    _, p = wilcoxon(sub["edc"].values, sub["edc_blb"].values)
    print(f"  {moi:<6}: n={len(sub):>3}  med_PLP={sub['edc'].median():.4f}  "
          f"med_BLB={sub['edc_blb'].median():.4f}  delta={sub['delta'].median():.4f}  p={p:.4f}")

# ── Export GraphPad CSVs ──────────────────────────────────────────────────────
print("\nExporting GraphPad CSVs...")
ORDER = ["AD","AR","XL","AD_AR"]

# Full P/LP EDC set (85 genes)
df_edc_plp_full = df_edc[df_edc["edc"].notna()].copy()

# Full B/LB EDC set (125 genes)
df_edc_blb_full = df_edc_blb[df_edc_blb["edc_blb"].notna()].copy()

# CSV 1: full per-gene paired table (82 genes)
df_edc_both.sort_values(["moi","edc"], ascending=[True,False]).to_csv(
    "nephvar_edc_per_gene_paired.csv", index=False)
print(f"  Saved: nephvar_edc_per_gene_paired.csv ({len(df_edc_both)} genes)")

# CSV 2: PLP EDC wide format — full 85-gene set
pd.DataFrame({f"PLP_{m}": pd.Series(df_edc_plp_full[df_edc_plp_full["moi"]==m]["edc"].values)
              for m in ORDER}).to_csv("nephvar_edc_by_moi_plp.csv", index=False)
print(f"  Saved: nephvar_edc_by_moi_plp.csv (AD={( df_edc_plp_full['moi']=='AD').sum()}, AR={(df_edc_plp_full['moi']=='AR').sum()}, XL={(df_edc_plp_full['moi']=='XL').sum()}, AD_AR={(df_edc_plp_full['moi']=='AD_AR').sum()})")

# CSV 3: BLB EDC wide format — full 125-gene set
pd.DataFrame({f"BLB_{m}": pd.Series(df_edc_blb_full[df_edc_blb_full["moi"]==m]["edc_blb"].values)
              for m in ORDER}).to_csv("nephvar_edc_by_moi_blb.csv", index=False)
print(f"  Saved: nephvar_edc_by_moi_blb.csv (AD={(df_edc_blb_full['moi']=='AD').sum()}, AR={(df_edc_blb_full['moi']=='AR').sum()}, XL={(df_edc_blb_full['moi']=='XL').sum()}, AD_AR={(df_edc_blb_full['moi']=='AD_AR').sum()})")

# CSV 4: paired PLP vs BLB per MOI (82-gene overlap)
for moi in ORDER:
    sub = df_edc_both[df_edc_both["moi"]==moi][["gene","edc","edc_blb","delta"]].copy()
    sub.to_csv(f"nephvar_edc_paired_{moi}.csv", index=False)
    print(f"  Saved: nephvar_edc_paired_{moi}.csv ({len(sub)} rows)")

# CSV 5: supplementary full table — all genes with any EDC
df_supp = df_edc_plp_full.merge(
    df_edc_blb_full[["gene","n_blb_residues","edc_blb"]], on="gene", how="outer"
).sort_values(["panel","moi","edc"], ascending=[True,True,False])
df_supp["delta"] = df_supp["edc"] - df_supp["edc_blb"]
df_supp.to_csv("nephvar_edc_supplementary.csv", index=False)
print(f"  Saved: nephvar_edc_supplementary.csv ({len(df_supp)} genes)")

# CSV 6: GraphPad grouped table PLP (gene as row label)
rows = []
for _, r in df_edc_plp_full.sort_values(["moi","edc"], ascending=[True,False]).iterrows():
    row = {"Gene": r["gene"]}
    for moi in ORDER:
        row[moi] = r["edc"] if r["moi"] == moi else ""
    rows.append(row)
pd.DataFrame(rows)[["Gene"]+ORDER].to_csv("nephvar_edc_graphpad_plp.csv", index=False)
print(f"  Saved: nephvar_edc_graphpad_plp.csv ({len(rows)} genes)")

# CSV 7: GraphPad grouped table BLB (gene as row label)
rows = []
for _, r in df_edc_blb_full.sort_values(["moi","edc_blb"], ascending=[True,False]).iterrows():
    row = {"Gene": r["gene"]}
    for moi in ORDER:
        row[moi] = r["edc_blb"] if r["moi"] == moi else ""
    rows.append(row)
pd.DataFrame(rows)[["Gene"]+ORDER].to_csv("nephvar_edc_graphpad_blb.csv", index=False)
print(f"  Saved: nephvar_edc_graphpad_blb.csv ({len(rows)} genes)")

print("\nDone.")

Computing P/LP EDC...
  P/LP EDC: 85 genes computed
Computing B/LB EDC...
  B/LB EDC: 125 genes computed
  Genes with both PLP+BLB EDC: 82

AD vs AR Mann-Whitney p=0.0000

MOI summary (P/LP EDC):
  AD    : n= 22  median=1.2808  IQR=[1.208,1.417]
  AR    : n= 48  median=1.0886  IQR=[0.999,1.183]
  XL    : n=  6  median=1.0285  IQR=[0.867,1.133]
  AD_AR : n=  9  median=0.9658  IQR=[0.942,1.205]

PLP vs BLB Wilcoxon by MOI:
  AD    : n= 20  med_PLP=1.2890  med_BLB=1.0041  delta=0.2777  p=0.0001
  AR    : n= 48  med_PLP=1.0886  med_BLB=1.0131  delta=0.1089  p=0.0106
  XL    : n=  5  med_PLP=0.9302  med_BLB=1.0648  delta=-0.1038  p=0.8125
  AD_AR : n=  9  med_PLP=0.9658  med_BLB=0.9532  delta=0.0246  p=0.3008

Exporting GraphPad CSVs...
  Saved: nephvar_edc_per_gene_paired.csv (82 genes)
  Saved: nephvar_edc_by_moi_plp.csv (AD=22, AR=48, XL=6, AD_AR=9)
  Saved: nephvar_edc_by_moi_blb.csv (AD=35, AR=76, XL=5, AD_AR=9)
  Saved: nephvar_edc_paired_AD.csv (20 rows)
  Saved: nephvar_edc_paired_A

# Examples

In [12]:
# ── Identify the clustering region in INF2 ────────────────────────────────────
import numpy as np

gene = "INF2"
gene_df = rsa_by_gene.get(gene, pd.DataFrame())

# Get P/LP variants with their details
plp_df = gene_df[gene_df["acmg"].isin(["P","LP"])].copy()
plp_df = plp_df.sort_values("resnum")

print(f"INF2 P/LP residues (n={len(plp_inf2)}):")
print(f"  Range: {min(plp_inf2)} — {max(plp_inf2)}")
print(f"  Median: {np.median(plp_inf2):.0f}")
print()

# Find contiguous clusters
clusters = []
current  = [plp_inf2[0]]
for r in plp_inf2[1:]:
    if r - current[-1] <= 20:   # within 20 residues = same cluster
        current.append(r)
    else:
        clusters.append(current)
        current = [r]
clusters.append(current)

print(f"Clusters (gap threshold = 20 residues):")
for i, cl in enumerate(clusters):
    print(f"  Cluster {i+1}: residues {min(cl)}–{max(cl)}  "
          f"(n={len(cl)}, span={max(cl)-min(cl)} aa)")

# Cross-reference with known INF2 domains
print(f"""
Known INF2 domain architecture:
  DID  (Diaphanous-inhibitory domain) : ~1–130
  GBD  (GTPase-binding domain)        : ~1–50
  Linker                              : ~130–400
  FH1  (Formin homology 1)            : ~400–600
  FH2  (Formin homology 2)            : ~600–1000
  CAAX (membrane targeting)           : ~1200–1249
  WH2  (DAD domain)                   : ~1070–1100

INF2 FSGS mutations cluster in DID (residues 1–130) —
disrupting DID-DAD autoinhibition → constitutively active formin → actin dysregulation
""")

# Cα distances within the main cluster
if len(clusters) > 0:
    main_cluster = max(clusters, key=len)
    coords = np.array([ca_coords[gene][r] for r in main_cluster if r in ca_coords[gene]])
    if len(coords) > 1:
        dists = []
        for i in range(len(coords)):
            for j in range(i+1, len(coords)):
                dists.append(np.sqrt(((coords[i]-coords[j])**2).sum()))
        print(f"Main cluster (residues {min(main_cluster)}–{max(main_cluster)}):")
        print(f"  Mean pairwise Cα distance: {np.mean(dists):.1f} Å")
        print(f"  Max pairwise Cα distance : {np.max(dists):.1f} Å")
        print(f"  This fits within a ~{np.max(dists):.0f} Å sphere")

INF2 P/LP residues (n=36):
  Range: 30 — 233
  Median: 112

Clusters (gap threshold = 20 residues):
  Cluster 1: residues 30–233  (n=36, span=203 aa)

Known INF2 domain architecture:
  DID  (Diaphanous-inhibitory domain) : ~1–130
  GBD  (GTPase-binding domain)        : ~1–50
  Linker                              : ~130–400
  FH1  (Formin homology 1)            : ~400–600
  FH2  (Formin homology 2)            : ~600–1000
  CAAX (membrane targeting)           : ~1200–1249
  WH2  (DAD domain)                   : ~1070–1100

INF2 FSGS mutations cluster in DID (residues 1–130) —
disrupting DID-DAD autoinhibition → constitutively active formin → actin dysregulation

Main cluster (residues 30–233):
  Mean pairwise Cα distance: 20.8 Å
  Max pairwise Cα distance : 46.2 Å
  This fits within a ~46 Å sphere


In [9]:
# ── NPHS1 — AR, low EDC (dispersed LoF) ──────────────────────────────────────
print("=" * 50)
print("NPHS1 — AR — LoF — EDC 1.184")
print("=" * 50)
view_nphs1, plp_nphs1, blb_nphs1 = make_edc_view(
    gene          = "NPHS1",
    cartoon_color = "#E8956D",
    plp_color     = "#E64B35",
    blb_color     = "#4DBBD5",
)
view_nphs1.show()

NPHS1 — AR — LoF — EDC 1.184
NPHS1  MOI=AR  EDC=1.184
  P/LP residues : 68  (red, large)
  B/LB residues : 39  (blue, small)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [13]:
# ── Identify clustering region in ACTN4 ───────────────────────────────────────
import numpy as np

gene = "ACTN4"
gene_df = rsa_by_gene.get(gene, pd.DataFrame())

plp_res = sorted(gene_df[gene_df["acmg"].isin(["P","LP"])]
                 ["resnum"].dropna().astype(int).unique().tolist())
blb_res = sorted(gene_df[gene_df["acmg"].isin(["B","LB"])]
                 ["resnum"].dropna().astype(int).unique().tolist())

print(f"ACTN4 P/LP residues (n={len(plp_res)}):")
print(f"  Range : {min(plp_res)} — {max(plp_res)}")
print(f"  Median: {np.median(plp_res):.0f}")
print(f"  List  : {plp_res}")

# Cluster detection
clusters = []
current  = [plp_res[0]]
for r in plp_res[1:]:
    if r - current[-1] <= 20:
        current.append(r)
    else:
        clusters.append(current)
        current = [r]
clusters.append(current)

print(f"\nClusters (gap threshold = 20 residues):")
for i, cl in enumerate(clusters):
    print(f"  Cluster {i+1}: residues {min(cl)}–{max(cl)}  "
          f"(n={len(cl)}, span={max(cl)-min(cl)} aa)")

# Cα distances for each cluster
for i, cl in enumerate(clusters):
    coords = np.array([ca_coords[gene][r] for r in cl if r in ca_coords[gene]])
    if len(coords) < 2:
        continue
    dists = []
    for a in range(len(coords)):
        for b in range(a+1, len(coords)):
            dists.append(np.sqrt(((coords[a]-coords[b])**2).sum()))
    print(f"\n  Cluster {i+1} 3D distances:")
    print(f"    Mean pairwise Cα : {np.mean(dists):.1f} Å")
    print(f"    Max pairwise Cα  : {np.max(dists):.1f} Å")

# Known ACTN4 domain architecture
print(f"""
Known ACTN4 domain architecture:
  ABD  (Actin-binding domain)     : ~1–260
    CH1 (Calponin homology 1)     : ~1–120
    CH2 (Calponin homology 2)     : ~121–260
  Spectrin repeats (SR1-4)        : ~270–740
  CaM  (Calmodulin-like domain)   : ~740–911

ACTN4 FSGS mutations cluster in ABD/CH2 —
disrupting actin-binding, increasing actin affinity → GoF
""")

ACTN4 P/LP residues (n=14):
  Range : 59 — 262
  Median: 184
  List  : [59, 72, 153, 165, 169, 173, 174, 195, 203, 240, 253, 255, 259, 262]

Clusters (gap threshold = 20 residues):
  Cluster 1: residues 59–72  (n=2, span=13 aa)
  Cluster 2: residues 153–174  (n=5, span=21 aa)
  Cluster 3: residues 195–203  (n=2, span=8 aa)
  Cluster 4: residues 240–262  (n=5, span=22 aa)

  Cluster 1 3D distances:
    Mean pairwise Cα : 11.8 Å
    Max pairwise Cα  : 11.8 Å

  Cluster 2 3D distances:
    Mean pairwise Cα : 11.7 Å
    Max pairwise Cα  : 22.1 Å

  Cluster 3 3D distances:
    Mean pairwise Cα : 12.6 Å
    Max pairwise Cα  : 12.6 Å

  Cluster 4 3D distances:
    Mean pairwise Cα : 9.7 Å
    Max pairwise Cα  : 14.7 Å

Known ACTN4 domain architecture:
  ABD  (Actin-binding domain)     : ~1–260
    CH1 (Calponin homology 1)     : ~1–120
    CH2 (Calponin homology 2)     : ~121–260
  Spectrin repeats (SR1-4)        : ~270–740
  CaM  (Calmodulin-like domain)   : ~740–911

ACTN4 FSGS mutations cl

In [11]:
print("=" * 50)
print("COL4A3 — AD/AR — LoF — EDC 0.860")
print("=" * 50)
view_col4a3, plp_col4a3, blb_col4a3 = make_edc_view(
    gene          = "COL4A3",
    cartoon_color = "#E8956D",
    plp_color     = "#E64B35",
    blb_color     = "#4DBBD5",
)
view_col4a3.show()

COL4A3 — AD/AR — LoF — EDC 0.860
COL4A3  MOI=AD_AR  EDC=0.860
  P/LP residues : 243  (red, large)
  B/LB residues : 51  (blue, small)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [14]:
# ── Quick GoF/clustering analysis for all AD genes ───────────────────────────
import numpy as np
import pandas as pd

ad_genes = df_edc[
    (df_edc["moi"] == "AD") &
    (df_edc["edc"].notna())
].sort_values("edc", ascending=False)

print(f"AD genes with EDC computed: {len(ad_genes)}")
print(f"\n{'gene':<12} {'panel':<6} {'edc':>6} {'n_plp':>6} {'res_range':<15} {'main_cluster':<20} {'known_mechanism'}")
print("-" * 100)

for _, row in ad_genes.iterrows():
    gene    = row["gene"]
    edc_val = row["edc"]
    panel   = row["panel"]

    gene_df = rsa_by_gene.get(gene, pd.DataFrame())
    plp_res = sorted(gene_df[gene_df["acmg"].isin(["P","LP"])]
                     ["resnum"].dropna().astype(int).unique().tolist())

    if len(plp_res) < 2:
        continue

    # Find main cluster
    clusters = []
    current  = [plp_res[0]]
    for r in plp_res[1:]:
        if r - current[-1] <= 25:
            current.append(r)
        else:
            clusters.append(current)
            current = [r]
    clusters.append(current)
    main_cl = max(clusters, key=len)

    # 3D compactness of main cluster
    coords = [ca_coords[gene][r] for r in main_cl if gene in ca_coords and r in ca_coords[gene]]
    if len(coords) >= 2:
        coords = np.array(coords)
        dists  = [np.sqrt(((coords[i]-coords[j])**2).sum())
                  for i in range(len(coords)) for j in range(i+1,len(coords))]
        max_d  = f"{max(dists):.0f}Å"
    else:
        max_d = "—"

    res_range    = f"{min(plp_res)}–{max(plp_res)}"
    cluster_str  = f"{min(main_cl)}–{max(main_cl)} (n={len(main_cl)}, {max_d})"

    print(f"{gene:<12} {panel:<6} {edc_val:>6.3f} {len(plp_res):>6} {res_range:<15} {cluster_str:<25}")

AD genes with EDC computed: 22

gene         panel     edc  n_plp res_range       main_cluster         known_mechanism
----------------------------------------------------------------------------------------------------
INF2         SRNS    2.018     36 30–233          30–233 (n=36, 46Å)       
GATA3        CAKUT   1.983     14 263–353         263–353 (n=14, 48Å)      
ACTN4        SRNS    1.937     14 59–262          153–203 (n=7, 26Å)       
LMX1B        SRNS    1.677     27 59–271          223–271 (n=14, 24Å)      
PAX2         CAKUT   1.608     28 24–328          24–140 (n=26, 64Å)       
SOX17        CAKUT   1.426      4 70–138          70–82 (n=2, 20Å)         
WT1          SRNS    1.387     44 62–429          62–280 (n=41, 107Å)      
HNF1B        CAKUT   1.360     58 1–547           74–173 (n=28, 73Å)       
UMOD         CAKUT   1.350     64 31–559          31–317 (n=59, 118Å)      
CASR         USD     1.295    100 1–971           95–245 (n=30, 57Å)       
FN1          CGN    

In [15]:
# ── Pull domain annotations from NephVar biophysical pages ───────────────────
import re, json, urllib.request
import pandas as pd

BIOPHYS_BASE = "https://nephvar.github.io/NephVar/biophysical"

def fetch_domains(gene):
    url = f"{BIOPHYS_BASE}/{gene}.html"
    try:
        with urllib.request.urlopen(url, timeout=15) as r:
            html = r.read().decode("utf-8", errors="replace")
        m = re.search(r'const domains\s*=\s*(\[.*?\]);', html, re.DOTALL)
        if not m:
            return []
        domains_json = m.group(1).replace(": NaN", ": null")
        return json.loads(domains_json)
    except Exception as e:
        print(f"  ERROR {gene}: {e}")
        return []

# ── For each AD gene, fetch domains and map P/LP clusters ─────────────────────
ad_genes_list = df_edc[
    (df_edc["moi"] == "AD") &
    (df_edc["edc"].notna())
].sort_values("edc", ascending=False)["gene"].tolist()

results = []
for gene in ad_genes_list:
    gene_df  = rsa_by_gene.get(gene, pd.DataFrame())
    plp_res  = sorted(gene_df[gene_df["acmg"].isin(["P","LP"])]
                      ["resnum"].dropna().astype(int).unique().tolist())
    if len(plp_res) < 2:
        continue

    domains  = fetch_domains(gene)
    edc_val  = df_edc[df_edc["gene"]==gene]["edc"].values[0]

    # Find clusters
    clusters = []
    current  = [plp_res[0]]
    for r in plp_res[1:]:
        if r - current[-1] <= 25:
            current.append(r)
        else:
            clusters.append(current)
            current = [r]
    clusters.append(current)
    main_cl = max(clusters, key=len)

    # Map main cluster residues to domains
    cluster_domains = set()
    for res in main_cl:
        for d in domains:
            if d.get("start") and d.get("end") and d.get("type") and d.get("description"):
                if d["type"] in {"Domain","Active site","Binding site","Motif","Region","Coiled coil","Transmembrane"}:
                    if d["start"] <= res <= d["end"]:
                        cluster_domains.add(d["description"])

    results.append({
        "gene"           : gene,
        "panel"          : gene_df["panel"].iloc[0] if len(gene_df) > 0 else "?",
        "edc"            : edc_val,
        "n_plp"          : len(plp_res),
        "plp_range"      : f"{min(plp_res)}–{max(plp_res)}",
        "main_cluster"   : f"{min(main_cl)}–{max(main_cl)} (n={len(main_cl)})",
        "cluster_domains": " | ".join(sorted(cluster_domains)) if cluster_domains else "—",
        "n_domains_total": len(domains),
    })

df_ad_domains = pd.DataFrame(results)

print(f"{'gene':<10} {'edc':>6} {'n_plp':>6}  {'main_cluster':<22} {'cluster_domains'}")
print("=" * 100)
for _, r in df_ad_domains.iterrows():
    print(f"{r['gene']:<10} {r['edc']:>6.3f} {r['n_plp']:>6}  "
          f"{r['main_cluster']:<22} {r['cluster_domains']}")

gene          edc  n_plp  main_cluster           cluster_domains
INF2        2.018     36  30–233 (n=36)          GBD/FH3
GATA3       1.983     14  263–353 (n=14)         Flexible linker | YxKxHxxxRP
ACTN4       1.937     14  153–203 (n=7)          Actin-binding | Calponin-homology (CH) 1 | Calponin-homology (CH) 2
LMX1B       1.677     27  223–271 (n=14)         Disordered
PAX2        1.608     28  24–140 (n=26)          PAI subdomain | RED subdomain
SOX17       1.426      4  70–82 (n=2)            —
WT1         1.387     44  62–280 (n=41)          9aaTAD | Disordered
HNF1B       1.360     58  74–173 (n=28)          Disordered | POU-specific atypical
UMOD        1.350     64  31–317 (n=59)          Beta hairpin | D10C | EGF-like 1 | EGF-like 2; calcium-binding | EGF-like 3; calcium-binding | EGF-like 4
CASR        1.295    100  95–245 (n=30)          Ligand-binding 1 (LB1) | Ligand-binding 2 (LB2)
FN1         1.283     16  213–260 (n=5)          Fibrin- and heparin-binding 1 | Fibrone

In [16]:
# ── Mechanistic interpretation from NephVar domain annotations ────────────────

interpretations = {
    "INF2"  : ("GoF",    "GBD/FH3 — mDia disinhibition → excess actin polymerization"),
    "GATA3" : ("DN",     "Flexible linker + YxKxHxxxRP — DNA-binding interface; DN on partner TFs"),
    "ACTN4" : ("GoF",    "CH1/CH2 actin-binding domain — CH1-CH2 hinge disruption → hyperbinding"),
    "LMX1B" : ("LoF/DN", "Disordered LIM domain region — haploinsufficiency; some DN on DNA binding"),
    "PAX2"  : ("LoF",    "PAI + RED subdomains — paired-box DNA-binding; haploinsufficiency"),
    "SOX17" : ("LoF",    "HMG-box — no domain hit but residues 70-82 in HMG; haploinsufficiency"),
    "WT1"   : ("GoF/DN", "9aaTAD + disordered — transcriptional activation domain; context-dependent"),
    "HNF1B" : ("LoF",    "POU-specific atypical domain — homeodomain DNA-binding; haploinsufficiency"),
    "UMOD"  : ("GoF",    "EGF-like + D10C + beta-hairpin — ER retention → protein aggregation → tubular injury"),
    "CASR"  : ("GoF",    "LB1 + LB2 ligand-binding domain — activating mutations → hypocalcemia/Bartter"),
    "FN1"   : ("GoF",    "FN type-I + heparin-binding — abnormal matrix deposition → glomerulopathy"),
    "MYH9"  : ("DN",     "Myosin motor + N-terminal SH3 — DN on myosin IIA filament assembly"),
    "SIX1"  : ("LoF/DN", "No domain hit — residues 106-139 likely in SIX domain; haploinsufficiency BOR"),
    "TRPC6" : ("GoF",    "No domain hit — residues 109-112 in ankyrin repeats; GoF Ca2+ influx"),
    "HNF4A" : ("LoF",    "NR LBD — nuclear receptor ligand-binding domain; haploinsufficiency MODY1"),
    "BMP4"  : ("LoF",    "No domain hit — too few variants (n=4); haploinsufficiency CAKUT"),
    "EYA1"  : ("LoF",    "No domain hit — residues 440-547 in EYA domain; haploinsufficiency BOR"),
    "VDR"   : ("LoF",    "NR LBD — nuclear receptor ligand-binding domain; haploinsufficiency"),
    "CFI"   : ("LoF",    "Peptidase S1 + SRCR + LDL-A — serine protease; haploinsufficiency aHUS/MPGN"),
    "RET"   : ("GoF/LoF","CLD3+CLD4 — cadherin-like domain; GoF→MEN2, LoF→HSCR; both in NephVar"),
    "TNXB"  : ("LoF",    "Single variant — insufficient for EDC; haploinsufficiency VUR"),
    "WNT4"  : ("LoF",    "Single variant — insufficient for EDC; haploinsufficiency CAKUT"),
}

print(f"{'gene':<10} {'edc':>6} {'mechanism':<10} {'domain_cluster':<45} {'interpretation'}")
print("=" * 130)
for _, r in df_ad_domains.iterrows():
    gene = r["gene"]
    mech, interp = interpretations.get(gene, ("?", "—"))
    print(f"{gene:<10} {r['edc']:>6.3f} {mech:<10} {r['cluster_domains']:<45} {interp}")

# ── EDC by mechanism ──────────────────────────────────────────────────────────
print(f"\nEDC by mechanism:")
df_ad_domains["mechanism"] = df_ad_domains["gene"].map(
    lambda g: interpretations.get(g, ("?",""))[0])

for mech in ["GoF","DN","GoF/DN","GoF/LoF","LoF/DN","LoF"]:
    sub = df_ad_domains[df_ad_domains["mechanism"]==mech]["edc"]
    if len(sub) == 0:
        continue
    genes_m = df_ad_domains[df_ad_domains["mechanism"]==mech]["gene"].tolist()
    print(f"  {mech:<10}: n={len(sub):>2}  median={sub.median():.3f}  "
          f"mean={sub.mean():.3f}  genes={genes_m}")

gene          edc mechanism  domain_cluster                                interpretation
INF2        2.018 GoF        GBD/FH3                                       GBD/FH3 — mDia disinhibition → excess actin polymerization
GATA3       1.983 DN         Flexible linker | YxKxHxxxRP                  Flexible linker + YxKxHxxxRP — DNA-binding interface; DN on partner TFs
ACTN4       1.937 GoF        Actin-binding | Calponin-homology (CH) 1 | Calponin-homology (CH) 2 CH1/CH2 actin-binding domain — CH1-CH2 hinge disruption → hyperbinding
LMX1B       1.677 LoF/DN     Disordered                                    Disordered LIM domain region — haploinsufficiency; some DN on DNA binding
PAX2        1.608 LoF        PAI subdomain | RED subdomain                 PAI + RED subdomains — paired-box DNA-binding; haploinsufficiency
SOX17       1.426 LoF        —                                             HMG-box — no domain hit but residues 70-82 in HMG; haploinsufficiency
WT1         1.387 GoF/DN  

In [18]:
# AD genes passing n>=10 threshold for mechanistic interpretation
df_ad_mech = df_ad_domains[df_ad_domains["n_plp"] >= 10].copy()

print(f"AD genes passing n>=10: {len(df_ad_mech)} / {len(df_ad_domains)}")
print(f"Excluded (n<10): {df_ad_domains[df_ad_domains['n_plp']<10]['gene'].tolist()}")
print()

print(f"{'gene':<10} {'edc':>6} {'n_plp':>6} {'mechanism':<10} {'cluster_domains'}")
print("=" * 90)
for _, r in df_ad_mech.iterrows():
    print(f"{r['gene']:<10} {r['edc']:>6.3f} {r['n_plp']:>6} "
          f"{r['mechanism']:<10} {r['cluster_domains']}")

print(f"\nEDC by mechanism (n>=10 only):")
for mech in ["GoF","DN","GoF/DN","GoF/LoF","LoF/DN","LoF"]:
    sub = df_ad_mech[df_ad_mech["mechanism"]==mech]["edc"]
    if len(sub) == 0:
        continue
    genes_m = df_ad_mech[df_ad_mech["mechanism"]==mech]["gene"].tolist()
    print(f"  {mech:<10}: n={len(sub):>2}  median={sub.median():.3f}  "
          f"genes={genes_m}")

AD genes passing n>=10: 18 / 22
Excluded (n<10): ['SOX17', 'BMP4', 'TNXB', 'WNT4']

gene          edc  n_plp mechanism  cluster_domains
INF2        2.018     36 GoF        GBD/FH3
GATA3       1.983     14 DN         Flexible linker | YxKxHxxxRP
ACTN4       1.937     14 GoF        Actin-binding | Calponin-homology (CH) 1 | Calponin-homology (CH) 2
LMX1B       1.677     27 LoF/DN     Disordered
PAX2        1.608     28 LoF        PAI subdomain | RED subdomain
WT1         1.387     44 GoF/DN     9aaTAD | Disordered
HNF1B       1.360     58 LoF        Disordered | POU-specific atypical
UMOD        1.350     64 GoF        Beta hairpin | D10C | EGF-like 1 | EGF-like 2; calcium-binding | EGF-like 3; calcium-binding | EGF-like 4
CASR        1.295    100 GoF        Ligand-binding 1 (LB1) | Ligand-binding 2 (LB2)
FN1         1.283     16 GoF        Fibrin- and heparin-binding 1 | Fibronectin type-I 4 | Fibronectin type-I 5
MYH9        1.278     27 DN         Mediates interaction with LIMCH1 | My

In [19]:
# ── 3D visualization — GATA3, LMX1B, PAX2 ────────────────────────────────────

genes_to_viz = [
    ("GATA3", "#9B59B6", "DN",     "EDC 1.983"),   # purple — DN
    ("LMX1B", "#E8956D", "LoF/DN", "EDC 1.677"),   # orange — LoF/DN
    ("PAX2",  "#4DBBD5", "LoF/DN", "EDC 1.608"),   # cyan — LoF/DN
]

for gene, cartoon_color, mech, edc_str in genes_to_viz:
    print("=" * 50)
    print(f"{gene} — CAKUT/SRNS — {mech} — {edc_str}")
    print("=" * 50)
    view, plp, blb = make_edc_view(
        gene          = gene,
        cartoon_color = cartoon_color,
        plp_color     = "#E64B35",
        blb_color     = "#4DBBD5",
    )
    if view:
        view.show()

GATA3 — CAKUT/SRNS — DN — EDC 1.983
GATA3  MOI=AD  EDC=1.983
  P/LP residues : 14  (red, large)
  B/LB residues : 4  (blue, small)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

LMX1B — CAKUT/SRNS — LoF/DN — EDC 1.677
LMX1B  MOI=AD  EDC=1.677
  P/LP residues : 27  (red, large)
  B/LB residues : 7  (blue, small)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

PAX2 — CAKUT/SRNS — LoF/DN — EDC 1.608
PAX2  MOI=AD  EDC=1.608
  P/LP residues : 28  (red, large)
  B/LB residues : 4  (blue, small)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.